The Bank Marketing dataset is a classic binary classification dataset used to predict whether a customer will subscribe to a bank term deposit (fastrenteinnskudd) (yes or no) based on direct phone marketing data.

In [2]:
from sklearn.datasets import fetch_openml

data = fetch_openml("bank-marketing", version=1, as_frame=True, parser="liac-arff")
df = data.frame
df.head()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,Class
0,58.0,management,married,tertiary,no,2143.0,yes,no,unknown,5.0,may,261.0,1.0,-1.0,0.0,unknown,1
1,44.0,technician,single,secondary,no,29.0,yes,no,unknown,5.0,may,151.0,1.0,-1.0,0.0,unknown,1
2,33.0,entrepreneur,married,secondary,no,2.0,yes,yes,unknown,5.0,may,76.0,1.0,-1.0,0.0,unknown,1
3,47.0,blue-collar,married,unknown,no,1506.0,yes,no,unknown,5.0,may,92.0,1.0,-1.0,0.0,unknown,1
4,33.0,unknown,single,unknown,no,1.0,no,no,unknown,5.0,may,198.0,1.0,-1.0,0.0,unknown,1


In [3]:
df.shape

(45211, 17)

In [ ]:
df.info() #Viser kolonnetyper og manglende verdier

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   V1      45211 non-null  float64 
 1   V2      45211 non-null  category
 2   V3      45211 non-null  category
 3   V4      45211 non-null  category
 4   V5      45211 non-null  category
 5   V6      45211 non-null  float64 
 6   V7      45211 non-null  category
 7   V8      45211 non-null  category
 8   V9      45211 non-null  category
 9   V10     45211 non-null  float64 
 10  V11     45211 non-null  category
 11  V12     45211 non-null  float64 
 12  V13     45211 non-null  float64 
 13  V14     45211 non-null  float64 
 14  V15     45211 non-null  float64 
 15  V16     45211 non-null  category
 16  Class   45211 non-null  category
dtypes: category(10), float64(7)
memory usage: 2.8 MB


Det vi ser er at alle kolonnene har 45 211 verdier, så vi slipper imputering (i statistikk er dette en prosess der man erstatter manglende data med estimater). Merk at ting kan likevel skjules i kategoriene, for eksempel verdien "unknown" (Det ser vi på senere).

In [6]:
df["Class"].value_counts() #Viser antall forekomster av hver klasse

Class
1    39922
2     5289
Name: count, dtype: int64

Klasse 1 (trolig "nei") er 39 922 rader, og klasse 2 ("ja") er 5 289 rader. Så ca. 11,7% har svart ja. Hvis man velger en modell som alltid svarer "nei" så får man dermed ca. 88.3% riktige uten å lære noe. Det betyr at accuracy (andelen riktige svar) er et misvisende mål her, og du må bruke noe som AUC, presisjon/recall eller F1.

In [7]:
navn = ["age", "job", "marital", "education", "default", "balance",
        "housing", "loan", "contact", "day", "month", "duration",
        "campaign", "pdays", "previous", "poutcome", "y"]
df.columns = navn
df["y"] = (df["y"].astype(int) == 2).astype(int)   # 1 = ja, 0 = nei
df["y"].mean()

np.float64(0.11698480458295547)

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier

X = df.drop(columns="y") #modellen skal predikere y basert på alle andre kolonner, og kan derfor ikke bruke y som input. Derfor dropper vi y fra X.
y = df["y"] #her er y det vi ønsker å predikere, og derfor er det outputen til modellen.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
) #deler opp datasettet i et treningssett og et testsett. Testsettet er 20% av datasettet, og stratify=y sørger for at fordelingen av klasser i y er den samme i trenings- og testsettet.

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train) #DummyClassifier ser ikke på kolonnene i det hele tatt. Med strategy="most_frequent" predikerer den alltid den klassen som forekommer oftest i treningssettet. Dette er en baseline-modell som vi kan sammenligne andre modeller med.
print("Accuracy:", dummy.score(X_test, y_test)) #Måle resultatet til baseline-modellen på testsettet. Accuracy er andelen riktige prediksjoner, og her vil den være lik andelen av den mest forekommende klassen i treningssettet.

Accuracy: 0.8830034280659074


In [ ]:
from sklearn.compose import ColumnTransformer 
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

kat = X.select_dtypes("category").columns
num = X.select_dtypes("number").columns

prep = ColumnTransformer([
    ("num", StandardScaler(), num),
    ("kat", OneHotEncoder(handle_unknown="ignore"), kat),
])

model = Pipeline([("prep", prep), ("lr", LogisticRegression(max_iter=1000))])
model.fit(X_train, y_train)

proba = model.predict_proba(X_test)[:, 1]
print("AUC med duration:", roc_auc_score(y_test, proba))

AUC med duration: 0.9055740146044154
